# 10. Trial-level Neural Inspection

Goal:
- Inspect raw data files for trial-level neural responses.
- Identify whether neural data exists as:
  - condition-level: units × conditions × time
  - trial-level: units × trials × time
  - session-level trial arrays
- Avoid loading huge files unless explicitly allowed.

In [1]:
from pathlib import Path
import sys
import types
import pickle
import gc
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
HAND_DMFC_DIR = RAW_DIR / "hand_dmfc"
NEURAL_DIR = RAW_DIR / "neural_responses"
RNN_DIR = RAW_DIR / "rnn"
PROCESSED_DIR = DATA_DIR / "processed"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("HAND_DMFC_DIR:", HAND_DMFC_DIR)
print("NEURAL_DIR:", NEURAL_DIR)
print("RNN_DIR:", RNN_DIR)

PROJECT_ROOT: c:\Users\ksbong\Documents\GitHub\ipe-pong-latents
RAW_DIR: c:\Users\ksbong\Documents\GitHub\ipe-pong-latents\data\raw
HAND_DMFC_DIR: c:\Users\ksbong\Documents\GitHub\ipe-pong-latents\data\raw\hand_dmfc
NEURAL_DIR: c:\Users\ksbong\Documents\GitHub\ipe-pong-latents\data\raw\neural_responses
RNN_DIR: c:\Users\ksbong\Documents\GitHub\ipe-pong-latents\data\raw\rnn


In [2]:
def file_table(root_dirs):
    rows = []
    
    for root in root_dirs:
        root = Path(root)
        if not root.exists():
            continue
        
        for p in root.rglob("*"):
            if p.is_file():
                rows.append({
                    "path": str(p.relative_to(PROJECT_ROOT)),
                    "name": p.name,
                    "suffix": p.suffix,
                    "size_mb": p.stat().st_size / 1024**2,
                })
    
    return pd.DataFrame(rows).sort_values("size_mb", ascending=False)


files_df = file_table([HAND_DMFC_DIR, NEURAL_DIR, RNN_DIR])
display(files_df)

,path,name,suffix,size_mb
8,data\raw\rnn\offline_rnn_neural_responses_reli...,offline_rnn_neural_responses_reliable_50.pkl,.pkl,2163.377455
0,data\raw\hand_dmfc\all_hand_dmfc_dataset_50ms.pkl,all_hand_dmfc_dataset_50ms.pkl,.pkl,1202.042446
2,data\raw\hand_dmfc\perle_hand_dmfc_dataset_50m...,perle_hand_dmfc_dataset_50ms.pkl,.pkl,1023.795490
1,data\raw\hand_dmfc\mahler_hand_dmfc_dataset_50...,mahler_hand_dmfc_dataset_50ms.pkl,.pkl,228.117346
6,data\raw\neural_responses\offline_mahler_hand_...,offline_mahler_hand_dmfc_neural_responses_reli...,.pkl,38.454469
5,data\raw\neural_responses\offline_all_hand_dmf...,offline_all_hand_dmfc_neural_responses_reliabl...,.pkl,38.454469
7,data\raw\neural_responses\offline_perle_hand_d...,offline_perle_hand_dmfc_neural_responses_relia...,.pkl,38.454469
3,data\raw\hand_dmfc\Source_Data.xlsx,Source_Data.xlsx,.xlsx,3.364403
9,data\raw\rnn\rnn_compare_all_hand_dmfc_occ_50m...,rnn_compare_all_hand_dmfc_occ_50ms_neural_resp...,.pkl,0.222986
4,data\raw\hand_dmfc\valid_meta_sample_full.pkl,valid_meta_sample_full.pkl,.pkl,0.070369


In [3]:
def patch_old_pandas_pickle():
    module_name = "pandas.core.indexes.numeric"
    
    if module_name not in sys.modules:
        numeric_module = types.ModuleType(module_name)
        numeric_module.Int64Index = pd.Index
        numeric_module.UInt64Index = pd.Index
        numeric_module.Float64Index = pd.Index
        sys.modules[module_name] = numeric_module


def load_pickle_compat(path: Path):
    patch_old_pandas_pickle()
    with open(path, "rb") as f:
        return pickle.load(f)

In [5]:
TARGET_N_TRIALS = 10158
TARGET_N_COND = 79
TARGET_N_TIME = 100

def is_interesting_shape(shape):
    shape = tuple(shape)
    
    has_trials = TARGET_N_TRIALS in shape
    has_cond = TARGET_N_COND in shape
    has_time = TARGET_N_TIME in shape
    
    # Trial-level 후보:
    # exact 10158이 있거나, 100 time axis와 큰 trial-like axis가 있음.
    has_trial_like_axis = any((s > 500 and s != TARGET_N_TIME) for s in shape)
    
    maybe_trial_neural = (
        (has_trials and has_time) or
        (has_time and has_trial_like_axis and len(shape) >= 2)
    )
    
    maybe_condition_neural = has_cond and has_time
    
    return {
        "has_trials_10158": has_trials,
        "has_cond_79": has_cond,
        "has_time_100": has_time,
        "has_trial_like_axis": has_trial_like_axis,
        "maybe_trial_neural": maybe_trial_neural,
        "maybe_condition_neural": maybe_condition_neural,
    }


def scan_object_shapes(obj, prefix="root", max_depth=8, depth=0, max_list_items=20):
    rows = []
    
    if depth > max_depth:
        return rows
    
    if isinstance(obj, np.ndarray):
        flags = is_interesting_shape(obj.shape)
        rows.append({
            "path": prefix,
            "type": "ndarray",
            "shape": obj.shape,
            "dtype": str(obj.dtype),
            "size_mb_est": obj.nbytes / 1024**2,
            **flags,
        })
    
    elif isinstance(obj, pd.DataFrame):
        flags = is_interesting_shape(obj.shape)
        rows.append({
            "path": prefix,
            "type": "DataFrame",
            "shape": obj.shape,
            "dtype": "mixed",
            "size_mb_est": obj.memory_usage(deep=True).sum() / 1024**2,
            "columns_preview": list(obj.columns[:15]),
            **flags,
        })
        
        # object columns can contain arrays/lists
        for c in obj.columns:
            if obj[c].dtype == "object":
                sample = obj[c].dropna()
                if len(sample) > 0:
                    val = sample.iloc[0]
                    if isinstance(val, (np.ndarray, list, dict)):
                        rows.extend(
                            scan_object_shapes(
                                val,
                                prefix=f"{prefix}/{c}[sample0]",
                                max_depth=max_depth,
                                depth=depth+1,
                                max_list_items=max_list_items,
                            )
                        )
    
    elif isinstance(obj, pd.Series):
        flags = is_interesting_shape(obj.shape)
        rows.append({
            "path": prefix,
            "type": "Series",
            "shape": obj.shape,
            "dtype": str(obj.dtype),
            "size_mb_est": obj.memory_usage(deep=True) / 1024**2,
            **flags,
        })
    
    elif isinstance(obj, dict):
        rows.append({
            "path": prefix,
            "type": "dict",
            "shape": f"len={len(obj)}",
            "dtype": "",
            "size_mb_est": np.nan,
            "keys_preview": list(obj.keys())[:20],
            **{
                "has_trials_10158": False,
                "has_cond_79": False,
                "has_time_100": False,
                "has_trial_like_axis": False,
                "maybe_trial_neural": False,
                "maybe_condition_neural": False,
            }
        })
        
        for k, v in obj.items():
            rows.extend(
                scan_object_shapes(
                    v,
                    prefix=f"{prefix}/{k}",
                    max_depth=max_depth,
                    depth=depth+1,
                    max_list_items=max_list_items,
                )
            )
    
    elif isinstance(obj, (list, tuple)):
        rows.append({
            "path": prefix,
            "type": type(obj).__name__,
            "shape": f"len={len(obj)}",
            "dtype": "",
            "size_mb_est": np.nan,
            "keys_preview": "",
            **{
                "has_trials_10158": False,
                "has_cond_79": False,
                "has_time_100": False,
                "has_trial_like_axis": False,
                "maybe_trial_neural": False,
                "maybe_condition_neural": False,
            }
        })
        
        for i, v in enumerate(obj[:max_list_items]):
            rows.extend(
                scan_object_shapes(
                    v,
                    prefix=f"{prefix}[{i}]",
                    max_depth=max_depth,
                    depth=depth+1,
                    max_list_items=max_list_items,
                )
            )
    
    else:
        # Ignore scalars / strings
        pass
    
    return rows

In [6]:
def scan_pickle_file(path, max_size_mb=500):
    path = Path(path)
    size_mb = path.stat().st_size / 1024**2
    
    if size_mb > max_size_mb:
        print(f"SKIP large pkl: {path.name} ({size_mb:.1f} MB)")
        return pd.DataFrame()
    
    print(f"\nLoading: {path.name} ({size_mb:.1f} MB)")
    obj = load_pickle_compat(path)
    
    rows = scan_object_shapes(obj, prefix=path.name, max_depth=8)
    df = pd.DataFrame(rows)
    df["file"] = path.name
    df["file_size_mb"] = size_mb
    
    del obj
    gc.collect()
    
    return df


pkl_files = list(HAND_DMFC_DIR.glob("*.pkl")) + list(NEURAL_DIR.glob("*.pkl")) + list(RNN_DIR.glob("*.pkl"))

# 우선 500MB 이하만
scan_dfs = []

for p in sorted(pkl_files):
    df_scan = scan_pickle_file(p, max_size_mb=500)
    if len(df_scan) > 0:
        scan_dfs.append(df_scan)

pkl_shape_scan = pd.concat(scan_dfs, ignore_index=True) if len(scan_dfs) > 0 else pd.DataFrame()

display(
    pkl_shape_scan.sort_values(
        ["maybe_trial_neural", "maybe_condition_neural", "size_mb_est"],
        ascending=[False, False, False],
    ).head(100)
)

SKIP large pkl: all_hand_dmfc_dataset_50ms.pkl (1202.0 MB)

Loading: mahler_hand_dmfc_dataset_50ms.pkl (228.1 MB)
SKIP large pkl: perle_hand_dmfc_dataset_50ms.pkl (1023.8 MB)

Loading: valid_meta_sample_full.pkl (0.1 MB)

Loading: offline_all_hand_dmfc_neural_responses_reliable_50.pkl (38.5 MB)

Loading: offline_mahler_hand_dmfc_neural_responses_reliable_50.pkl (38.5 MB)

Loading: offline_perle_hand_dmfc_neural_responses_reliable_50.pkl (38.5 MB)
SKIP large pkl: offline_rnn_neural_responses_reliable_50.pkl (2163.4 MB)

Loading: rnn_compare_all_hand_dmfc_occ_50ms_neural_responses_reliable_FactorAnalysis_50.pkl (0.2 MB)


,path,type,shape,dtype,size_mb_est,keys_preview,has_trials_10158,has_cond_79,has_time_100,has_trial_like_axis,maybe_trial_neural,maybe_condition_neural,columns_preview,file,file_size_mb
5,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(518, 79, 100)",float64,31.221008,NaN,False,True,True,True,True,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
8,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(518, 79, 100)",float64,31.221008,NaN,False,True,True,True,True,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
11,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(518, 79, 100)",float64,31.221008,NaN,False,True,True,True,True,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
650,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(337, 79, 100)",float64,20.311737,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
652,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(337, 79, 100)",float64,20.311737,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
654,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(337, 79, 100)",float64,20.311737,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
677,offline_all_hand_dmfc_neural_responses_reliabl...,ndarray,"(100, 79, 100)",float64,6.027222,NaN,False,True,True,False,False,True,NaN,offline_all_hand_dmfc_neural_responses_reliabl...,38.454469
678,offline_all_hand_dmfc_neural_responses_reliabl...,ndarray,"(100, 79, 100)",float64,6.027222,NaN,False,True,True,False,False,True,NaN,offline_all_hand_dmfc_neural_responses_reliabl...,38.454469
690,offline_all_hand_dmfc_neural_responses_reliabl...,ndarray,"(100, 79, 100)",float64,6.027222,NaN,False,True,True,False,False,True,NaN,offline_all_hand_dmfc_neural_responses_reliabl...,38.454469
691,offline_all_hand_dmfc_neural_responses_reliabl...,ndarray,"(100, 79, 100)",float64,6.027222,NaN,False,True,True,False,False,True,NaN,offline_all_hand_dmfc_neural_responses_reliabl...,38.454469


In [7]:
if len(pkl_shape_scan) > 0:
    trial_candidates = pkl_shape_scan[
        pkl_shape_scan["maybe_trial_neural"].fillna(False)
    ].copy()
    
    cond_candidates = pkl_shape_scan[
        pkl_shape_scan["maybe_condition_neural"].fillna(False)
    ].copy()
    
    print("Trial-level neural candidates:")
    display(
        trial_candidates.sort_values("size_mb_est", ascending=False)
    )
    
    print("\nCondition-level neural candidates:")
    display(
        cond_candidates.sort_values("size_mb_est", ascending=False)
    )
else:
    print("No scan results.")

Trial-level neural candidates:


,path,type,shape,dtype,size_mb_est,keys_preview,has_trials_10158,has_cond_79,has_time_100,has_trial_like_axis,maybe_trial_neural,maybe_condition_neural,columns_preview,file,file_size_mb
5,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(518, 79, 100)",float64,31.221008,NaN,False,True,True,True,True,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
8,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(518, 79, 100)",float64,31.221008,NaN,False,True,True,True,True,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
11,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(518, 79, 100)",float64,31.221008,NaN,False,True,True,True,True,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346



Condition-level neural candidates:


,path,type,shape,dtype,size_mb_est,keys_preview,has_trials_10158,has_cond_79,has_time_100,has_trial_like_axis,maybe_trial_neural,maybe_condition_neural,columns_preview,file,file_size_mb
5,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(518, 79, 100)",float64,31.221008,NaN,False,True,True,True,True,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
8,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(518, 79, 100)",float64,31.221008,NaN,False,True,True,True,True,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
11,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(518, 79, 100)",float64,31.221008,NaN,False,True,True,True,True,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
652,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(337, 79, 100)",float64,20.311737,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
654,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(337, 79, 100)",float64,20.311737,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41,mahler_hand_dmfc_dataset_50ms.pkl/behavioral_r...,ndarray,"(79, 100)",float64,0.060272,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
42,mahler_hand_dmfc_dataset_50ms.pkl/behavioral_r...,ndarray,"(79, 100)",float64,0.060272,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
43,mahler_hand_dmfc_dataset_50ms.pkl/behavioral_r...,ndarray,"(79, 100)",float64,0.060272,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
44,mahler_hand_dmfc_dataset_50ms.pkl/behavioral_r...,ndarray,"(79, 100)",float64,0.060272,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346


In [8]:
MAHLER_PKL = HAND_DMFC_DIR / "mahler_hand_dmfc_dataset_50ms.pkl"

mahler_scan = scan_pickle_file(MAHLER_PKL, max_size_mb=1000)

display(
    mahler_scan.sort_values(
        ["maybe_trial_neural", "maybe_condition_neural", "size_mb_est"],
        ascending=[False, False, False],
    ).head(100)
)


Loading: mahler_hand_dmfc_dataset_50ms.pkl (228.1 MB)


,path,type,shape,dtype,size_mb_est,keys_preview,has_trials_10158,has_cond_79,has_time_100,has_trial_like_axis,maybe_trial_neural,maybe_condition_neural,columns_preview,file,file_size_mb
5,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(518, 79, 100)",float64,31.221008,NaN,False,True,True,True,True,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
8,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(518, 79, 100)",float64,31.221008,NaN,False,True,True,True,True,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
11,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(518, 79, 100)",float64,31.221008,NaN,False,True,True,True,True,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
650,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(337, 79, 100)",float64,20.311737,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
652,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(337, 79, 100)",float64,20.311737,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
654,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(337, 79, 100)",float64,20.311737,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
6,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(83, 79, 100)",float64,5.002594,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
9,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(83, 79, 100)",float64,5.002594,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
12,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(83, 79, 100)",float64,5.002594,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
668,mahler_hand_dmfc_dataset_50ms.pkl/neural_respo...,ndarray,"(50, 79, 100)",float64,3.013611,NaN,False,True,True,False,False,True,NaN,mahler_hand_dmfc_dataset_50ms.pkl,228.117346


In [9]:
data = load_pickle_compat(MAHLER_PKL)

print("Top-level keys:")
for k in data.keys():
    print(" -", k)

for key in data.keys():
    if "neural" in str(key).lower() or "response" in str(key).lower():
        print("\nKEY:", key)
        val = data[key]
        print("type:", type(val))
        if isinstance(val, dict):
            print("subkeys:", val.keys())
            for sk, sv in val.items():
                if isinstance(sv, np.ndarray):
                    print(f"  {sk}: ndarray shape={sv.shape}, dtype={sv.dtype}")
                elif isinstance(sv, dict):
                    print(f"  {sk}: dict keys={list(sv.keys())[:20]}")
                else:
                    print(f"  {sk}: {type(sv)}")
        elif isinstance(val, np.ndarray):
            print("shape:", val.shape)
        elif isinstance(val, pd.DataFrame):
            print("shape:", val.shape)
            print("columns:", list(val.columns[:30]))

Top-level keys:
 - trial_meta
 - neural_meta
 - neural_responses
 - neural_responses_sh1
 - neural_responses_sh2
 - behavioral_responses
 - masks
 - ntr_vis
 - ntr_occ
 - meta
 - neural_idx_global
 - conds_ch_dict
 - reliable_neural_idx
 - neural_responses_reliable
 - neural_responses_reliable_sh1
 - neural_responses_reliable_sh2
 - neural_responses_reliable_FactorAnalysis_10
 - neural_responses_reliable_FactorAnalysis_10_sh1
 - neural_responses_reliable_FactorAnalysis_10_sh2
 - neural_responses_reliable_FactorAnalysis_20
 - neural_responses_reliable_FactorAnalysis_20_sh1
 - neural_responses_reliable_FactorAnalysis_20_sh2
 - neural_responses_reliable_FactorAnalysis_50
 - neural_responses_reliable_FactorAnalysis_50_sh1
 - neural_responses_reliable_FactorAnalysis_50_sh2

KEY: neural_meta
type: <class 'pandas.DataFrame'>
shape: (518, 33)
columns: ['cluster_idx', 'templates', 'num_stable_trials', 'cluster_location', 'ncond_occ_alpha == 1 & success == 1', 'ncond_sh1_occ_alpha == 1 & success

In [14]:
# Search every .mat file under data/raw recursively
mat_files = list(RAW_DIR.rglob("*.mat"))

print("Number of MAT files found:", len(mat_files))
for p in mat_files:
    print(" -", p.relative_to(PROJECT_ROOT), f"({p.stat().st_size / 1024**2:.1f} MB)")

print("Number of MAT files found:", len(mat_files))
for p in mat_files:
    print(" -", p, f"({p.stat().st_size / 1024**2:.1f} MB)")
mat_rows = []

for p in mat_files:
    size_mb = p.stat().st_size / 1024**2
    print(f"\nScanning MAT: {p.relative_to(PROJECT_ROOT)} ({size_mb:.1f} MB)")
    
    scanned = False
    
    try:
        import scipy.io as sio
        info = sio.whosmat(p)
        
        print(f"  scipy.whosmat variables: {len(info)}")
        
        for name, shape, dtype in info:
            flags = is_interesting_shape(shape)
            mat_rows.append({
                "file": str(p.relative_to(PROJECT_ROOT)),
                "var": name,
                "shape": shape,
                "dtype": dtype,
                "file_size_mb": size_mb,
                "method": "scipy.whosmat",
                **flags,
            })
        
        scanned = True
    
    except Exception as e:
        print("  scipy.whosmat failed:", repr(e))
    
    if not scanned:
        try:
            import h5py
            
            h5_count = [0]
            
            with h5py.File(p, "r") as f:
                def visit_func(name, obj):
                    if hasattr(obj, "shape"):
                        shape = tuple(obj.shape)
                        flags = is_interesting_shape(shape)
                        
                        mat_rows.append({
                            "file": str(p.relative_to(PROJECT_ROOT)),
                            "var": name,
                            "shape": shape,
                            "dtype": str(getattr(obj, "dtype", "")),
                            "file_size_mb": size_mb,
                            "method": "h5py",
                            **flags,
                        })
                        
                        h5_count[0] += 1
                
                f.visititems(visit_func)
            
            print(f"  h5py datasets: {h5_count[0]}")
        
        except Exception as e2:
            print("  h5py scan failed:", repr(e2))

if len(mat_rows) == 0:
    mat_shape_scan = pd.DataFrame(columns=[
        "file", "var", "shape", "dtype", "file_size_mb", "method",
        "has_trials_10158", "has_cond_79", "has_time_100",
        "has_trial_like_axis", "maybe_trial_neural", "maybe_condition_neural",
    ])
else:
    mat_shape_scan = pd.DataFrame(mat_rows)

display(
    mat_shape_scan.sort_values(
        ["maybe_trial_neural", "maybe_condition_neural", "file_size_mb"],
        ascending=[False, False, False],
    ).head(200)
)

Number of MAT files found: 3
 - data\raw\glm\PVAF_allocentric.mat (0.2 MB)
 - data\raw\glm\run_glm.mat (520.1 MB)
 - data\raw\glm\run_glm_egocentric_ball.mat (520.9 MB)
Number of MAT files found: 3
 - c:\Users\ksbong\Documents\GitHub\ipe-pong-latents\data\raw\glm\PVAF_allocentric.mat (0.2 MB)
 - c:\Users\ksbong\Documents\GitHub\ipe-pong-latents\data\raw\glm\run_glm.mat (520.1 MB)
 - c:\Users\ksbong\Documents\GitHub\ipe-pong-latents\data\raw\glm\run_glm_egocentric_ball.mat (520.9 MB)

Scanning MAT: data\raw\glm\PVAF_allocentric.mat (0.2 MB)
  scipy.whosmat variables: 2

Scanning MAT: data\raw\glm\run_glm.mat (520.1 MB)
  scipy.whosmat failed: TypeError("'NoneType' object is not iterable")
  h5py scan failed: OSError('Unable to synchronously open file (file signature not found)')

Scanning MAT: data\raw\glm\run_glm_egocentric_ball.mat (520.9 MB)
  scipy.whosmat failed: TypeError("'NoneType' object is not iterable")
  h5py scan failed: OSError('Unable to synchronously open file (file sign

,file,var,shape,dtype,file_size_mb,method,has_trials_10158,has_cond_79,has_time_100,has_trial_like_axis,maybe_trial_neural,maybe_condition_neural
0,data\raw\glm\PVAF_allocentric.mat,PVAF_occ3,"(2576, 7)",double,0.197671,scipy.whosmat,False,False,False,True,False,False
1,data\raw\glm\PVAF_allocentric.mat,PVAF_vis3,"(1385, 7)",double,0.197671,scipy.whosmat,False,False,False,True,False,False


In [15]:
print("MAT trial-level candidates:")
display(
    mat_shape_scan[mat_shape_scan["maybe_trial_neural"].fillna(False)]
    .sort_values("file_size_mb", ascending=False)
)

print("MAT condition-level candidates:")
display(
    mat_shape_scan[mat_shape_scan["maybe_condition_neural"].fillna(False)]
    .sort_values("file_size_mb", ascending=False)
)

MAT trial-level candidates:


,file,var,shape,dtype,file_size_mb,method,has_trials_10158,has_cond_79,has_time_100,has_trial_like_axis,maybe_trial_neural,maybe_condition_neural


MAT condition-level candidates:


,file,var,shape,dtype,file_size_mb,method,has_trials_10158,has_cond_79,has_time_100,has_trial_like_axis,maybe_trial_neural,maybe_condition_neural


In [16]:
pkl_files = list(RAW_DIR.rglob("*.pkl"))

pkl_file_df = pd.DataFrame([
    {
        "path": str(p.relative_to(PROJECT_ROOT)),
        "name": p.name,
        "size_mb": p.stat().st_size / 1024**2,
    }
    for p in pkl_files
]).sort_values("size_mb", ascending=False)

display(pkl_file_df)

,path,name,size_mb
11,data\raw\rnn\offline_rnn_neural_responses_reli...,offline_rnn_neural_responses_reliable_50.pkl,2163.377455
0,data\raw\decode\decode_all_hand_dmfc_ego_occ_s...,decode_all_hand_dmfc_ego_occ_start_end_pad0_50...,1858.215078
1,data\raw\decode\decode_all_hand_dmfc_occ_start...,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,1295.293944
3,data\raw\decode\decode_perle_hand_dmfc_occ_sta...,decode_perle_hand_dmfc_occ_start_end_pad0_50ms...,1295.293944
2,data\raw\decode\decode_mahler_hand_dmfc_occ_st...,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,1295.293944
4,data\raw\hand_dmfc\all_hand_dmfc_dataset_50ms.pkl,all_hand_dmfc_dataset_50ms.pkl,1202.042446
6,data\raw\hand_dmfc\perle_hand_dmfc_dataset_50m...,perle_hand_dmfc_dataset_50ms.pkl,1023.795490
5,data\raw\hand_dmfc\mahler_hand_dmfc_dataset_50...,mahler_hand_dmfc_dataset_50ms.pkl,228.117346
8,data\raw\neural_responses\offline_all_hand_dmf...,offline_all_hand_dmfc_neural_responses_reliabl...,38.454469
10,data\raw\neural_responses\offline_perle_hand_d...,offline_perle_hand_dmfc_neural_responses_relia...,38.454469


In [17]:
scan_dfs = []

for p in sorted(pkl_files):
    # 일단 1300MB까지 scan. 2GB RNN은 아직 skip.
    df_scan = scan_pickle_file(p, max_size_mb=1300)
    if len(df_scan) > 0:
        scan_dfs.append(df_scan)

pkl_shape_scan = pd.concat(scan_dfs, ignore_index=True) if len(scan_dfs) > 0 else pd.DataFrame()

display(
    pkl_shape_scan.sort_values(
        ["maybe_trial_neural", "maybe_condition_neural", "size_mb_est"],
        ascending=[False, False, False],
    ).head(200)
)

SKIP large pkl: decode_all_hand_dmfc_ego_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl (1858.2 MB)

Loading: decode_all_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl (1295.3 MB)

Loading: decode_mahler_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl (1295.3 MB)

Loading: decode_perle_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl (1295.3 MB)

Loading: all_hand_dmfc_dataset_50ms.pkl (1202.0 MB)

Loading: mahler_hand_dmfc_dataset_50ms.pkl (228.1 MB)

Loading: perle_hand_dmfc_dataset_50ms.pkl (1023.8 MB)

Loading: valid_meta_sample_full.pkl (0.1 MB)

Loading: offline_all_hand_dmfc_neural_responses_reliable_50.pkl (38.5 MB)

Loading: offline_mahler_hand_dmfc_neural_responses_reliable_50.pkl (38.5 MB)

Loading: offline_perle_hand_dmfc_neural_responses_reliable_50.pkl (38.5 MB)
SKIP large pkl: offline_rnn_neural_responses_reliable_50.pkl (2163

,path,type,shape,dtype,size_mb_est,keys_preview,has_trials_10158,has_cond_79,has_time_100,has_trial_like_axis,maybe_trial_neural,maybe_condition_neural,file,file_size_mb,columns_preview
713,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(2576, 79, 100)",float64,155.261230,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
716,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(2576, 79, 100)",float64,155.261230,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
719,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(2576, 79, 100)",float64,155.261230,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
2063,perle_hand_dmfc_dataset_50ms.pkl/neural_respon...,ndarray,"(2058, 79, 100)",float64,124.040222,NaN,False,True,True,True,True,True,perle_hand_dmfc_dataset_50ms.pkl,1023.795490,NaN
2066,perle_hand_dmfc_dataset_50ms.pkl/neural_respon...,ndarray,"(2058, 79, 100)",float64,124.040222,NaN,False,True,True,True,True,True,perle_hand_dmfc_dataset_50ms.pkl,1023.795490,NaN
2069,perle_hand_dmfc_dataset_50ms.pkl/neural_respon...,ndarray,"(2058, 79, 100)",float64,124.040222,NaN,False,True,True,True,True,True,perle_hand_dmfc_dataset_50ms.pkl,1023.795490,NaN
1362,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(1889, 79, 100)",float64,113.854218,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
1364,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(1889, 79, 100)",float64,113.854218,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
1366,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(1889, 79, 100)",float64,113.854218,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
2708,perle_hand_dmfc_dataset_50ms.pkl/neural_respon...,ndarray,"(1552, 79, 100)",float64,93.542480,NaN,False,True,True,True,True,True,perle_hand_dmfc_dataset_50ms.pkl,1023.795490,NaN


In [18]:
trial_candidates = pkl_shape_scan[
    pkl_shape_scan["maybe_trial_neural"].fillna(False)
].copy()

cond_candidates = pkl_shape_scan[
    pkl_shape_scan["maybe_condition_neural"].fillna(False)
].copy()

print("PKL trial-level candidates:")
display(
    trial_candidates.sort_values("size_mb_est", ascending=False)
)

print("PKL condition-level candidates:")
display(
    cond_candidates.sort_values("size_mb_est", ascending=False)
)

PKL trial-level candidates:


,path,type,shape,dtype,size_mb_est,keys_preview,has_trials_10158,has_cond_79,has_time_100,has_trial_like_axis,maybe_trial_neural,maybe_condition_neural,file,file_size_mb,columns_preview
716,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(2576, 79, 100)",float64,155.261230,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
713,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(2576, 79, 100)",float64,155.261230,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
719,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(2576, 79, 100)",float64,155.261230,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
2063,perle_hand_dmfc_dataset_50ms.pkl/neural_respon...,ndarray,"(2058, 79, 100)",float64,124.040222,NaN,False,True,True,True,True,True,perle_hand_dmfc_dataset_50ms.pkl,1023.795490,NaN
2069,perle_hand_dmfc_dataset_50ms.pkl/neural_respon...,ndarray,"(2058, 79, 100)",float64,124.040222,NaN,False,True,True,True,True,True,perle_hand_dmfc_dataset_50ms.pkl,1023.795490,NaN
2066,perle_hand_dmfc_dataset_50ms.pkl/neural_respon...,ndarray,"(2058, 79, 100)",float64,124.040222,NaN,False,True,True,True,True,True,perle_hand_dmfc_dataset_50ms.pkl,1023.795490,NaN
1366,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(1889, 79, 100)",float64,113.854218,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
1362,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(1889, 79, 100)",float64,113.854218,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
1364,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(1889, 79, 100)",float64,113.854218,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
2712,perle_hand_dmfc_dataset_50ms.pkl/neural_respon...,ndarray,"(1552, 79, 100)",float64,93.542480,NaN,False,True,True,True,True,True,perle_hand_dmfc_dataset_50ms.pkl,1023.795490,NaN


PKL condition-level candidates:


,path,type,shape,dtype,size_mb_est,keys_preview,has_trials_10158,has_cond_79,has_time_100,has_trial_like_axis,maybe_trial_neural,maybe_condition_neural,file,file_size_mb,columns_preview
719,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(2576, 79, 100)",float64,155.261230,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
716,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(2576, 79, 100)",float64,155.261230,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
713,all_hand_dmfc_dataset_50ms.pkl/neural_response...,ndarray,"(2576, 79, 100)",float64,155.261230,NaN,False,True,True,True,True,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
2069,perle_hand_dmfc_dataset_50ms.pkl/neural_respon...,ndarray,"(2058, 79, 100)",float64,124.040222,NaN,False,True,True,True,True,True,perle_hand_dmfc_dataset_50ms.pkl,1023.795490,NaN
2063,perle_hand_dmfc_dataset_50ms.pkl/neural_respon...,ndarray,"(2058, 79, 100)",float64,124.040222,NaN,False,True,True,True,True,True,perle_hand_dmfc_dataset_50ms.pkl,1023.795490,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
768,all_hand_dmfc_dataset_50ms.pkl/behavioral_resp...,ndarray,"(79, 100)",float64,0.060272,NaN,False,True,True,False,False,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
769,all_hand_dmfc_dataset_50ms.pkl/behavioral_resp...,ndarray,"(79, 100)",float64,0.060272,NaN,False,True,True,False,False,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
770,all_hand_dmfc_dataset_50ms.pkl/behavioral_resp...,ndarray,"(79, 100)",float64,0.060272,NaN,False,True,True,False,False,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN
771,all_hand_dmfc_dataset_50ms.pkl/behavioral_resp...,ndarray,"(79, 100)",float64,0.060272,NaN,False,True,True,False,False,True,all_hand_dmfc_dataset_50ms.pkl,1202.042446,NaN


In [19]:
decode_candidates = pkl_shape_scan[
    pkl_shape_scan["path"].astype(str).str.contains("decode", case=False, na=False)
].copy()

decode_candidates = decode_candidates[
    decode_candidates["shape"].astype(str).str.contains("100", na=False)
].copy()

display(
    decode_candidates.sort_values(
        ["maybe_trial_neural", "size_mb_est"],
        ascending=[False, False],
    ).head(100)
)

,path,type,shape,dtype,size_mb_est,keys_preview,has_trials_10158,has_cond_79,has_time_100,has_trial_like_axis,maybe_trial_neural,maybe_condition_neural,file,file_size_mb,columns_preview
118,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,ndarray,"(100, 3502, 23)",float64,61.451721,NaN,False,False,True,True,True,False,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,1295.293944,NaN
121,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,ndarray,"(100, 3502, 23)",float64,61.451721,NaN,False,False,True,True,True,False,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,1295.293944,NaN
354,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3502, 23)",float64,61.451721,NaN,False,False,True,True,True,False,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,1295.293944,NaN
357,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3502, 23)",float64,61.451721,NaN,False,False,True,True,True,False,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,1295.293944,NaN
590,decode_perle_hand_dmfc_occ_start_end_pad0_50ms...,ndarray,"(100, 3502, 23)",float64,61.451721,NaN,False,False,True,True,True,False,decode_perle_hand_dmfc_occ_start_end_pad0_50ms...,1295.293944,NaN
593,decode_perle_hand_dmfc_occ_start_end_pad0_50ms...,ndarray,"(100, 3502, 23)",float64,61.451721,NaN,False,False,True,True,True,False,decode_perle_hand_dmfc_occ_start_end_pad0_50ms...,1295.293944,NaN
98,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,ndarray,"(100, 3423, 23)",float64,60.065460,NaN,False,False,True,True,True,False,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,1295.293944,NaN
101,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,ndarray,"(100, 3423, 23)",float64,60.065460,NaN,False,False,True,True,True,False,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,1295.293944,NaN
138,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,ndarray,"(100, 3423, 23)",float64,60.065460,NaN,False,False,True,True,True,False,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,1295.293944,NaN
141,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,ndarray,"(100, 3423, 23)",float64,60.065460,NaN,False,False,True,True,True,False,decode_all_hand_dmfc_occ_start_end_pad0_50ms_0...,1295.293944,NaN


In [20]:
mahler_decode_candidates = decode_candidates[
    decode_candidates["file"].astype(str).str.contains("mahler", case=False, na=False)
].copy()

display(
    mahler_decode_candidates[
        ["file", "path", "shape", "dtype", "size_mb_est"]
    ].sort_values("size_mb_est", ascending=False).head(100)
)

for p in mahler_decode_candidates["path"].head(30):
    print(p)

,file,path,shape,dtype,size_mb_est
357,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,"(100, 3502, 23)",float64,61.451721
354,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,"(100, 3502, 23)",float64,61.451721
334,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,"(100, 3423, 23)",float64,60.065460
337,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,"(100, 3423, 23)",float64,60.065460
374,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,"(100, 3423, 23)",float64,60.065460
377,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,"(100, 3423, 23)",float64,60.065460
317,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,"(100, 3344, 23)",float64,58.679199
397,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,"(100, 3344, 23)",float64,58.679199
394,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,"(100, 3344, 23)",float64,58.679199
314,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,"(100, 3344, 23)",float64,58.679199


decode_mahler_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl/res_decode[0]/r_dist
decode_mahler_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl/res_decode[0]/rs_dist
decode_mahler_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl/res_decode[0]/mae_dist
decode_mahler_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl/res_decode[0]/y_pred_dist
decode_mahler_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl/res_decode[0]/y_true_dist
decode_mahler_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl/res_decode[1]/r_dist
decode_mahler_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl/res_decode[1]/rs_dist
decode_mahler_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl/res_decode[1]/mae_dist
decode_m

In [21]:
decode_file_name = (
    mahler_decode_candidates
    .sort_values("size_mb_est", ascending=False)
    ["file"]
    .iloc[0]
)

decode_file_path = RAW_DIR / "decode" / decode_file_name

print("decode_file_name:", decode_file_name)
print("decode_file_path:", decode_file_path)
print("exists:", decode_file_path.exists())

decode_file_name: decode_mahler_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl
decode_file_path: c:\Users\ksbong\Documents\GitHub\ipe-pong-latents\data\raw\decode\decode_mahler_hand_dmfc_occ_start_end_pad0_50ms_0.50_neural_responses_reliable_FactorAnalysis_50.pkl
exists: True


In [22]:
decode_obj = load_pickle_compat(decode_file_path)

print("type:", type(decode_obj))

if isinstance(decode_obj, dict):
    print("top-level keys:")
    for k in decode_obj.keys():
        print(" -", k, type(decode_obj[k]))
else:
    print(repr(decode_obj)[:500])

type: <class 'dict'>
top-level keys:
 - res_decode <class 'list'>
 - ncond <class 'int'>
 - mask_conditions <class 'list'>
 - condition <class 'str'>
 - neural_data_to_use  <class 'str'>
 - beh_to_decode <class 'list'>
 - decoder_specs <class 'dict'>


In [23]:
def summarize_obj(obj, name="root", max_depth=2, depth=0):
    indent = "  " * depth
    
    if isinstance(obj, pd.DataFrame):
        print(f"{indent}{name}: DataFrame shape={obj.shape}")
        print(f"{indent}  columns={list(obj.columns)[:30]}")
    
    elif isinstance(obj, np.ndarray):
        print(f"{indent}{name}: ndarray shape={obj.shape}, dtype={obj.dtype}")
    
    elif isinstance(obj, dict):
        print(f"{indent}{name}: dict len={len(obj)}")
        keys = list(obj.keys())
        print(f"{indent}  keys={keys[:30]}")
        
        if depth < max_depth:
            for k in keys[:30]:
                summarize_obj(obj[k], name=str(k), max_depth=max_depth, depth=depth+1)
    
    elif isinstance(obj, list):
        print(f"{indent}{name}: list len={len(obj)}")
        if len(obj) > 0 and depth < max_depth:
            summarize_obj(obj[0], name=f"{name}[0]", max_depth=max_depth, depth=depth+1)
    
    else:
        print(f"{indent}{name}: {type(obj).__name__} {repr(obj)[:100]}")

In [24]:
summarize_obj(decode_obj, max_depth=3)

root: dict len=7
  keys=['res_decode', 'ncond', 'mask_conditions', 'condition', 'neural_data_to_use ', 'beh_to_decode', 'decoder_specs']
  res_decode: list len=11
    res_decode[0]: dict len=16
      keys=['unflatten_res', 'r_mu', 'r_sd', 'r_dist', 'rs_mu', 'rs_sd', 'rs_dist', 'mae_mu', 'mae_sd', 'mae_dist', 'y_pred_mu', 'y_pred_sd', 'y_pred_dist', 'y_true_mu', 'y_true_sd', 'y_true_dist']
      unflatten_res: dict len=3
        keys=['X', 'idx', 's']
      r_mu: ndarray shape=(23,), dtype=float64
      r_sd: ndarray shape=(23,), dtype=float64
      r_dist: ndarray shape=(100, 23), dtype=float64
      rs_mu: ndarray shape=(23,), dtype=float64
      rs_sd: ndarray shape=(23,), dtype=float64
      rs_dist: ndarray shape=(100, 23), dtype=float64
      mae_mu: ndarray shape=(23,), dtype=float64
      mae_sd: ndarray shape=(23,), dtype=float64
      mae_dist: ndarray shape=(100, 23), dtype=float64
      y_pred_mu: ndarray shape=(3107, 23), dtype=float64
      y_pred_sd: ndarray shape=(3107, 

In [25]:
decode_scan_rows = scan_object_shapes(
    decode_obj,
    prefix=decode_file_name,
    max_depth=10,
)

decode_scan_df = pd.DataFrame(decode_scan_rows)

display(
    decode_scan_df.sort_values(
        ["maybe_trial_neural", "size_mb_est"],
        ascending=[False, False],
    ).head(100)
)

target_shape_rows = decode_scan_df[
    decode_scan_df["shape"].astype(str).str.contains("3502|3423|3344|3265|3186|3107", na=False)
].copy()

display(
    target_shape_rows[
        ["path", "type", "shape", "dtype", "size_mb_est", "maybe_trial_neural", "maybe_condition_neural"]
    ].sort_values("size_mb_est", ascending=False)
)

,path,type,shape,dtype,size_mb_est,keys_preview,has_trials_10158,has_cond_79,has_time_100,has_trial_like_axis,maybe_trial_neural,maybe_condition_neural
118,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3502, 23)",float64,61.451721,NaN,False,False,True,True,True,False
121,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3502, 23)",float64,61.451721,NaN,False,False,True,True,True,False
98,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3423, 23)",float64,60.065460,NaN,False,False,True,True,True,False
101,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3423, 23)",float64,60.065460,NaN,False,False,True,True,True,False
138,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3423, 23)",float64,60.065460,NaN,False,False,True,True,True,False
141,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3423, 23)",float64,60.065460,NaN,False,False,True,True,True,False
78,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3344, 23)",float64,58.679199,NaN,False,False,True,True,True,False
81,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3344, 23)",float64,58.679199,NaN,False,False,True,True,True,False
158,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3344, 23)",float64,58.679199,NaN,False,False,True,True,True,False
161,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3344, 23)",float64,58.679199,NaN,False,False,True,True,True,False


,path,type,shape,dtype,size_mb_est,maybe_trial_neural,maybe_condition_neural
121,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3502, 23)",float64,61.451721,True,False
118,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3502, 23)",float64,61.451721,True,False
101,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3423, 23)",float64,60.065460,True,False
98,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3423, 23)",float64,60.065460,True,False
138,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3423, 23)",float64,60.065460,True,False
141,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3423, 23)",float64,60.065460,True,False
78,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3344, 23)",float64,58.679199,True,False
81,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3344, 23)",float64,58.679199,True,False
161,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3344, 23)",float64,58.679199,True,False
158,decode_mahler_hand_dmfc_occ_start_end_pad0_50m...,ndarray,"(100, 3344, 23)",float64,58.679199,True,False
